In [1]:
import pandas 
import requests
from bs4 import BeautifulSoup
import json
import numpy as np
import openai
import torch
import os
from sentence_transformers import SentenceTransformer

c:\Users\anoop\anaconda3\envs\Deep\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

True

In [11]:
# Path to the directory containing the extracted HTML files
docs_dir = "./pandas"

# Function to parse all HTML files in the directory
def parse_local_docs(docs_dir):
    docs = []
    for root, _, files in os.walk(docs_dir):
        for file in files:
            if file.endswith(".html"):  # Only process HTML files
                file_path = os.path.join(root, file)
                with open(file_path, "r", encoding="utf-8") as f:
                    soup = BeautifulSoup(f, "html.parser")
                    title = soup.title.string if soup.title else file
                    content = soup.get_text(separator="\n")  # Extract text content
                    docs.append({"title": title, "content": content})
    return docs

# Parse the local documentation
docs = parse_local_docs(docs_dir)

# Save parsed documentation to a JSON file
with open("pandas_docs_local.json", "w") as f:
    json.dump(docs, f)

print(f"Parsed {len(docs)} HTML files from the local documentation.")

Parsed 3849 HTML files from the local documentation.


In [12]:
def preprocess_docs(docs, chunk_size=500):
    chunks = []
    for doc in docs:
        content = doc["content"]
        for i in range(0, len(content), chunk_size):
            chunk = content[i : i + chunk_size]
            chunks.append({"title": doc["title"], "chunk": chunk})
    return chunks

# Preprocess the parsed documents
preprocessed_docs = preprocess_docs(docs)

# Save the preprocessed documents
with open("preprocessed_docs_local.json", "w") as f:
    json.dump(preprocessed_docs, f)

In [13]:
# Load preprocessed documents
with open("preprocessed_docs_local.json", "r") as f:
    preprocessed_docs = json.load(f)

# Generate embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer("all-MiniLM-L6-v2").to(device)
embeddings = model.encode(
    [chunk["chunk"] for chunk in preprocessed_docs], convert_to_tensor=True, device=device
)

# Save embeddings
np.save("embeddings_local.npy", embeddings.cpu().numpy())

c:\Users\anoop\anaconda3\envs\Deep\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anoop\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
# Load preprocessed documents
with open("preprocessed_docs_local.json", "r") as f:
    preprocessed_docs = json.load(f)

# Initialize the embedding model and move it to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer("all-MiniLM-L6-v2").to(device)

# Generate embeddings on GPU
def generate_embeddings(docs, model, batch_size=32):
    """
    Generate embeddings for the given documents on GPU.

    Args:
        docs (list): List of document chunks.
        model (SentenceTransformer): The embedding model.
        batch_size (int): Batch size for embedding generation.

    Returns:
        np.array: Generated embeddings.
    """
    embeddings = []
    for i in range(0, len(docs), batch_size):
        batch = [doc["chunk"] for doc in docs[i:i + batch_size]]
        batch_embeddings = model.encode(batch, convert_to_tensor=True, device=device)
        embeddings.append(batch_embeddings.cpu().numpy())
    return np.vstack(embeddings)

# Generate and save embeddings
embeddings = generate_embeddings(preprocessed_docs, model)
np.save("embeddings_local_gpu.npy", embeddings)

In [4]:
from torch.nn.functional import cosine_similarity

def retrieve_top_k(query, model, embeddings, docs, k=3):
    """
    Retrieve the top-k most relevant chunks for the given query using GPU-accelerated similarity search.

    Args:
        query (str): The user query.
        model (SentenceTransformer): The embedding model.
        embeddings (np.array): Precomputed document embeddings.
        docs (list): Preprocessed document chunks.
        k (int): Number of top documents to retrieve.

    Returns:
        list: Top-k document chunks.
    """
    # Generate query embedding on GPU
    query_embedding = model.encode(query, convert_to_tensor=True, device=device)

    # Move embeddings to GPU
    embeddings_tensor = torch.tensor(embeddings, device=device)

    # Calculate cosine similarity
    similarities = cosine_similarity(query_embedding.unsqueeze(0), embeddings_tensor)

    # Get top-k indices
    top_k_indices = torch.topk(similarities, k=k).indices.cpu().numpy()

    # Retrieve corresponding documents
    return [docs[i] for i in top_k_indices]